In [ ]:
# 0. 패키지 버전 맞추기 — Colab에서 처음 한 번: 이 셀 실행 → 런타임 다시 시작 → Run All
import sys
if 'google.colab' in sys.modules:
    !pip install -q pandas==3.1.0 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 xgboost==3.4.1 lightgbm==4.7.0 pyarrow==26.0.0 PyYAML==6.0.3
    print('설치 끝: 처음이면 런타임 → 세션 다시 시작 후 Run All')
else:
    print('로컬: 설치 생략')

# model_test — 기본 피처 확장: XGBoost (Colab GPU)

설정은 `config/experiment.yaml`의 `BASE_FEATURES.model_input.xgboost`. **Run All**로 실행한다.

- **입력 세트 6개**: 기본 / +H1 / +H3 / +H1+H3 / +H2 / +H2+H4 (`src/evaluation/base_model_runs.input_sets`).
- **결측**: 관측값으로 매달 변환한 뒤 빈 칸 0(= 그 달 평균), 학습·예측 모두. **교수님 지침의 예외(2026-11-21 사용자 결정, 확인 전)**. 근거 Chen·McCoy 2024 JFE, GKX 2020. 목표값은 채우지 않음. 목표월 2001-01·02는 학습에서 뺌(시장 변동성 12개월 창).
- **중간 저장·재시작**: 단위가 끝날 때마다 `<결과 위치>/<AUTHOR>/<RUN_NAME>_parts/`에 저장. 끊기면 커널 재시작 후 **Run All**만 다시 하면 저장된 단위는 건너뛰고 이어간다. 설정·코드·입력이 저장 때와 다르면 섞이지 않게 멈춘다(그때는 RUN_NAME을 바꿔 새로 시작). 결과 폴더 저장도 한 번만 한다.
- **XGBoost**: 회귀 4(윈저·z × MSE·Huber②, L2 0) + pairwise·순위회귀 각 L2 0·5, 깊이 3, 학습률 0.01·0.05, 트리 수 해마다 선택과 500 고정(hparams 끝 `_t500`), 입력 순위. 장치 GPU(cuda). 종류마다 결과 폴더 하나.
- **Drive 준비**: `data/processed/`에 `base_features.parquet`, `h2.parquet`(hybrid), `h4_epu_current.parquet`가 있어야 한다(1번 셀이 확인).

In [ ]:
import shutil, subprocess
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
else:
    raise RuntimeError('GPU 런타임이 아니다(XGBoost는 cuda로 돌린다)')

## 1. 설정 (Drive 연결·코드 받기)

In [ ]:
import os, sys, subprocess
from datetime import datetime
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/JiunShin1214/finance_analytics.git'
GIT_REF = '49a7895'             # Colab: 코드 버전 고정(커밋 해시). 재시작해도 같은 코드 → 저장 단위 이어 쓰기 가능
AUTHOR = 'jiun'                 # 결과 폴더 model_results/<AUTHOR>/ (영문·숫자·_·-만)
RUN_NAME = 'basefeat_v2'        # 같은 이름으로 다시 Run All 하면 Drive에 저장된 단위를 이어 씀
DRIVE_BASE = Path('/content/drive/MyDrive/금융데이터인텔리전스')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    if not GIT_REF:
        raise RuntimeError("GIT_REF에 브랜치 이름을 적는다.")
    ROOT = Path('/content/finance_analytics')
    if not ROOT.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(ROOT)], check=True)
    subprocess.run(['git', '-C', str(ROOT), 'fetch', 'origin', '--prune'], check=True)
    _remote = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--verify', '--quiet', f'origin/{GIT_REF}'],
                             capture_output=True, text=True).stdout.strip()
    subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', _remote or GIT_REF], check=True)
    print('받은 코드:', GIT_REF, '@', subprocess.run(['git', '-C', str(ROOT), 'rev-parse', '--short', 'HEAD'],
                                               capture_output=True, text=True, check=True).stdout.strip())
    if not (ROOT / 'data').is_symlink():       # 저장소 data/를 Drive 공유 data로 연결
        (ROOT / 'data').rename(ROOT / 'data_repo_placeholder')
        (ROOT / 'data').symlink_to(DRIVE_BASE / 'data')
    OUT_ROOT = Path('/content/drive/MyDrive/model_results')   # 중간 저장(parts)과 결과 모두 내 드라이브 → 세션이 끊겨도 남음
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src').is_dir() and (p / 'config' / 'experiment.yaml').is_file())
    OUT_ROOT = None

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
import pandas as pd
from src.evaluation import model_test as mt
from src.evaluation import base_model_runs as bmr
from src.features.h3 import compute_h3_sigma_grid

config = yaml.safe_load((ROOT / 'config' / 'experiment.yaml').read_text(encoding='utf-8'))
mt.check_no_test(config)
need = [config['PATHS'][k] for k in ('panel', 'eligibility', 'h1_components', 'base_features', 'h2', 'h4')]
missing = [n for n in need if not (ROOT / n).is_file()]
if missing:
    raise FileNotFoundError('Drive data/에 없음: ' + ', '.join(missing))
STARTED = datetime.now()
PARTS = bmr.parts_dir(ROOT, config, author=AUTHOR, run_name=RUN_NAME, out_root=OUT_ROOT)
print('중간 저장 폴더:', PARTS)
print('이미 저장된 단위:', len(list(PARTS.glob('*.pkl'))))

In [ ]:
# H3 σ 후보(λ 10개)는 한 번만 계산해 전처리에 같이 쓴다
panel = pd.read_parquet(ROOT / config['PATHS']['panel'], columns=['stock_id', 'month', 'return', 'eligible'])
eligibility = pd.read_parquet(ROOT / config['PATHS']['eligibility'])
H3_GRID = compute_h3_sigma_grid(panel, eligibility)
del panel, eligibility
print('H3 σ 격자:', H3_GRID.shape)

## 2. XGBoost (세트 6 × 단위 8 = 48단위)

In [ ]:
XGB_PREP = bmr.prepare(ROOT, config, 'boosting', input_transform='rank', h3_grid=H3_GRID)
fp = bmr.fingerprint(ROOT, config, 'boosting', 'rank')
XGB_RESULTS = bmr.run_units(bmr.boosting_units(XGB_PREP, config, 'xgboost'), PARTS, fp)

In [ ]:
for r in bmr.finalize_boosting(ROOT, config, XGB_PREP, XGB_RESULTS, author=AUTHOR, run_name=RUN_NAME, started=STARTED,
                               parts=PARTS, out_root=OUT_ROOT):
    print(r)